# Lunar DEM Profile Comparison

This notebook compares terrain profiles extracted from three DEMs:

1. The initial relative-height DEM reconstructed using Shape-from-Shading.
2. The reference DEM containing elevation data for the same terrain region.
3. The calibrated DEM produced using linear regression.

The three rasters must cover the same terrain region and be spatially aligned. The reference and calibrated DEMs must use consistent elevation units.

The reference data are not included as a ready-to-use dataset in this repository. Prepare the required files before running the profile comparison.

## 1. Load and Validate the DEMs

Load the initial relative DEM, reference DEM and calibrated DEM from
TIFF files.

Check that the input files exist and that the three arrays have
matching dimensions before performing profile comparisons.

In [ ]:
from pathlib import Path
import imageio.v2 as imageio
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates

dem_paths = {
    "Initial SFS DEM": Path("initial_predicted_dem.tiff"),
    "Reference DEM": Path("reference_dem.tif"),
    "Calibrated DEM": Path("predicted_final_absolute_dem.tiff"),
}

dems = {}

for name, path in dem_paths.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} was not found. "
            "Check the required inputs and run the preceding notebooks "
            "where applicable."
        )

    dem = imageio.imread(path).astype(np.float32)

    if dem.ndim != 2:
        raise ValueError(f"{name} must be a single-channel 2D raster.")

    dems[name] = dem
    valid = np.isfinite(dem)

    print(f"{name}: shape = {dem.shape}")

    if np.any(valid):
        print(
            f"  Valid value range: {dem[valid].min():.4f} "
            f"to {dem[valid].max():.4f}"
        )
    else:
        raise ValueError(f"{name} contains no finite values.")

shapes = {dem.shape for dem in dems.values()}

if len(shapes) != 1:
    raise ValueError(
        "The three DEMs have different shapes. "
        "Prepare them on the same grid before comparing profiles."
    )

print("All DEM array shapes match.")
print("Verify spatial alignment before interpreting the profiles.")

## 2. Interactive Profile-Line Selection

Display the reference DEM and select two points that define the
terrain profile to be analysed.

The selected pixel coordinates are used to sample all three DEMs
along the same line.

In [ ]:
def select_profile_line(dem_display):
    """
    Display the reference DEM and collect two points for profile extraction.

    Returns
    -------
    list
        Two selected (x, y) pixel coordinates.
    """
    points = []

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(dem_display, cmap="terrain")
    ax.set_title("Click two points to define the profile line")
    ax.set_xlabel("X Coordinate (pixels)")
    ax.set_ylabel("Y Coordinate (pixels)")

    def onclick(event):
        if event.inaxes != ax or event.xdata is None or event.ydata is None:
            return

        if len(points) >= 2:
            return

        points.append((event.xdata, event.ydata))

        ax.plot(
            event.xdata,
            event.ydata,
            "r+",
            markersize=12,
            markeredgewidth=2
        )

        print(
            f"Selected point {len(points)}: "
            f"({event.xdata:.2f}, {event.ydata:.2f})"
        )

        if len(points) == 2:
            ax.plot(
                [points[0][0], points[1][0]],
                [points[0][1], points[1][1]],
                "r--",
                linewidth=2
            )
            print("Both points selected.")
            fig.canvas.draw_idle()
            plt.pause(0.5)
            plt.close(fig)
        else:
            fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", onclick)

    print("Click two points on the reference DEM.")
    plt.show()

    if len(points) != 2:
        raise RuntimeError(
            "Two points were not selected. "
            "Use an interactive Matplotlib backend and try again."
        )

    return points

## Manual Profile Selection

If interactive mouse selection is unavailable, enter the start and end coordinates of the profile manually.

Coordinates are given as `(x, y)`, where `x` is the column index and `y` is the row index of the DEM.

In [ ]:
def select_profile_manually(dem, start_point, end_point):
    """
    Select a profile using (x, y) pixel coordinates.
    """
    height, width = dem.shape

    for name, point in [
        ("start_point", start_point),
        ("end_point", end_point),
    ]:
        x, y = point

        if not (0 <= x < width and 0 <= y < height):
            raise ValueError(
                f"{name}={point} is outside the DEM dimensions "
                f"(width={width}, height={height})."
            )

    return [start_point, end_point]


# Example coordinates only. Change these to select your own profile.
# Coordinates are (x, y), not (row, column).
start_point = (100, 100)
end_point = (900, 900)

profile_points = select_profile_manually(
    dems["Reference DEM"],
    start_point,
    end_point,
)

print("Selected profile:", profile_points)

## 3. Generate Comparative Height Profiles

Sample the initial SFS DEM, reference DEM and calibrated DEM along
the selected line using interpolation.

Plot the reference and calibrated elevations together, with the
initial relative SFS profile on a secondary vertical axis.

In [ ]:
def generate_profile_plot(dems, points):
    """
    Extract and plot height profiles along the line defined by two points.
    """
    start_point = np.asarray(points[0], dtype=float)
    end_point = np.asarray(points[1], dtype=float)

    line_length = np.linalg.norm(end_point - start_point)

    if line_length < 1:
        raise ValueError("The selected points are too close together.")

    num_points_on_line = int(line_length) + 1

    x_line = np.linspace(
        start_point[0], end_point[0], num_points_on_line
    )
    y_line = np.linspace(
        start_point[1], end_point[1], num_points_on_line
    )

    # map_coordinates expects (row, column), corresponding to (y, x)
    coords = np.vstack((y_line, x_line))

    # Sample each DEM along the same pixel-space line
    profile_ref = map_coordinates(
        dems["Reference DEM"],
        coords,
        order=1,
        mode="nearest"
    )
    profile_calibrated = map_coordinates(
        dems["Calibrated DEM"],
        coords,
        order=1,
        mode="nearest"
    )
    profile_sfs = map_coordinates(
        dems["Initial SFS DEM"],
        coords,
        order=1,
        mode="nearest"
    )


    distance = np.linspace(0, line_length, num_points_on_line)

    # Absolute elevations: reference and calibrated DEM
    fig, ax1 = plt.subplots(figsize=(14, 7))

    ax1.plot(
        distance,
        profile_ref,
        linestyle="--",
        linewidth=2.5,
        label="Reference DEM"
    )

    ax1.plot(
        distance,
        profile_calibrated,
        linewidth=2,
        label="Calibrated SFS DEM"
    )

    ax1.set_xlabel("Distance Along Profile (pixels)")
    ax1.set_ylabel("Absolute Elevation (reference units)")
    ax1.grid(True, linestyle=":", alpha=0.7)

    # Relative SFS heights: separate vertical axis
    ax2 = ax1.twinx()

    ax2.plot(
        distance,
        profile_sfs,
        color="crimson",
        linewidth=2,
        label="Initial SFS DEM (Relative)"
    )

    ax2.set_ylabel("Initial SFS Height (relative units)")

    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        loc="best"
    )

    fig.suptitle("Comparative Lunar DEM Height Profile")
    fig.tight_layout()
    plt.show()

    return {
        "distance": distance,
        "reference": profile_ref,
        "calibrated": profile_calibrated,
        "initial_sfs": profile_sfs
    }

## 4. Run the Comparative Profile Analysis

Select two points on the reference DEM, extract the corresponding
profiles from all three DEMs, and display the comparison.

In [ ]:
# Use the manual coordinates above, or select points interactively.
# To use interactive selection instead, uncomment the next line:
# profile_points = select_profile_line(dems["Reference DEM"])

profiles = generate_profile_plot(dems, profile_points)